# Exercise 03: Data Cleaning and Missing Values in NCII Harm Reporting
## Cluster 4 - Pandas and Visualization
**Framework:** NIST AI RMF Measure Function, EU AI Act Article 52

**Case Study:** NCII

In [8]:
n = 100

data = {
    'report_id': np.arange(1, n+1),
    'platform': np.random.choice(['Instagram', 'Facebook', 'Twitter'], n),
    'consent_violated': np.random.choice(['Yes', 'No'], n),
    'image_description': np.random.choice(['explicit image', 'face swap', 'doctored photo', None], n),
    'harm_description': np.random.choice(['emotional distress', 'job loss', 'relationship damage', 'public harassment', None], n),
}

df = pd.DataFrame(data)
print(df.head(10))
print(df.isnull().sum())

   report_id  platform consent_violated image_description     harm_description
0          1  Facebook               No    explicit image                 None
1          2  Facebook               No         face swap             job loss
2          3  Facebook               No    explicit image    public harassment
3          4  Facebook              Yes    explicit image  relationship damage
4          5   Twitter              Yes    doctored photo                 None
5          6   Twitter              Yes    doctored photo    public harassment
6          7   Twitter              Yes    explicit image                 None
7          8  Facebook              Yes              None   emotional distress
8          9   Twitter              Yes              None  relationship damage
9         10  Facebook               No         face swap   emotional distress
report_id             0
platform              0
consent_violated      0
image_description    27
harm_description     20
dtype: int6

In [9]:
print(df.groupby('consent_violated')['harm_description'].apply(lambda x: x.isnull().sum()))

consent_violated
No     10
Yes    10
Name: harm_description, dtype: int64


In [10]:
print(df.groupby('consent_violated')['image_description'].apply(lambda x: x.isnull().sum()))

consent_violated
No     18
Yes     9
Name: image_description, dtype: int64


In [11]:
df_flagged = df.copy()
df_flagged['image_description_missing'] = df_flagged['image_description'].isnull().astype(int)
df_flagged['harm_description_missing'] = df_flagged['harm_description'].isnull().astype(int)
print(df_flagged.head(10))

   report_id  platform consent_violated image_description  \
0          1  Facebook               No    explicit image   
1          2  Facebook               No         face swap   
2          3  Facebook               No    explicit image   
3          4  Facebook              Yes    explicit image   
4          5   Twitter              Yes    doctored photo   
5          6   Twitter              Yes    doctored photo   
6          7   Twitter              Yes    explicit image   
7          8  Facebook              Yes              None   
8          9   Twitter              Yes              None   
9         10  Facebook               No         face swap   

      harm_description  image_description_missing  harm_description_missing  
0                 None                          0                         1  
1             job loss                          0                         0  
2    public harassment                          0                         0  
3  relationship 

**Exercise 03 Governance Reflection**

Flagging missing values rather than dropping them is essential in NCII reporting systems. Optional fields were deliberately optional to protect reporter privacy — removing reports that skip them punishes reporters for exercising that choice. A woman who skips the harm description field because it is retraumatizing should not have her report silently discarded; the image still exists and still causes harm.

Under EU AI Act Article 52, platforms must be transparent about how AI systems process data. Flagging creates an auditable record: a regulator can see exactly which reports had missing fields and identify meaningful patterns. In this dataset, missing image descriptions concentrated in reports where consent was claimed not to be violated (18 missing vs 9 missing) — suggesting those reporters may have been testing the system, confused about definitions, or reporting on behalf of someone else. This pattern is governance-relevant information that would be permanently lost if those rows were dropped.

Under NIST AI RMF Measure function, measuring means quantifying not just what data exists but what is absent and why. Missing data is not a technical inconvenience — it is a signal. Flagging preserves that signal for human review rather than erasing it from the audit trail. Dropping data silently is a transparency failure; flagging it is a governance commitment.

